# IsoME – Estimate / check the input parameters

A pre-flight check for an `arguments` input struct, before committing to a full
`EliashbergSolver` / `RealAxisSolver` run:

1. **Read in** the inputs (`checkInput`, `InputParser`) and **plot everything that was read**:
   α²F(ω), N(ε) and W(ε,ε′) — whichever the chosen mode provides.
2. **Inspect the Coulomb parameters**: how μ*_ME follows from μ*_AD / μ, and how it reacts to
   the frequency cutoff.
3. **Run a single self-consistency step** in the selected mode and plot the resulting
   Z, Δ, χ (vDOS only) starting from the BCS initial guess.
4. **Inspect f(μ)** = N_e^nsc − N_e^sc, the function whose root the μ-update solves (`mu_flag = 1`).

One iteration is *not* a converged solution — Δ and Z here are one step away from the initial
guess. The point is to see whether the grids, cutoffs and kernels are sane (finite, smooth,
right order of magnitude) and whether the μ-update has a well-behaved root, which is cheap to
check and is where most bad parameter choices show up.

Every physics call goes into the IsoME package (non-exported functions as `IsoME.<function>`);
the cells mirror `solve_eliashberg()` in `src/TcSearch.jl` and `solve_realAxis_cDOS()` /
`solve_realAxis_vDOS()` in `src/realAxisSolver.jl`, with the iteration loop cut to one pass.

In [ ]:
# ----- Environment setup -----
# Assumes the kernel is started inside scripts/, so the package root is one level up.
import Pkg
pkg_dir = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(pkg_dir)

using IsoME
using Plots
using Trapz
using Interpolations

default(show=false)

## Step 1) Inputs

`axis` selects which solver the single iteration is taken from:

| `axis` | solver | equations |
|---|---|---|
| `:matsubara` | `EliashbergSolver` | `IsoME.eliashberg_eqn` (`src/EliashbergEq.jl`) |
| `:real` | `RealAxisSolver` | `IsoME.realEliashbergEq` (`src/realAxisEliashbergEq.jl`) |

The mode itself (cDOS / vDOS, ±W) comes from `inp` as usual: `cDOS_flag` and `include_Weep`.
The default below is the Nb vDOS+W test case, which exercises every plot in this notebook
(a2F + DOS + W, χ, and the μ-update). For a cDOS+μ* run, drop `dos_file` / `Weep_file`
and set `cDOS_flag = 1`, `include_Weep = 0`; the cells below skip what is not available.

Note the real-axis kernel precompute is by far the most expensive step here — on the real axis
prefer a modest `reOmega_c` / a coarse `domega` for a quick check.

In [ ]:
axis = :matsubara       # :matsubara (EliashbergSolver) or :real (RealAxisSolver)

# Seed of the vDOS iteration:
#   false -> single step from the flat BCS guess (fast, what this notebook is for)
#   true  -> converged cDOS solution as seed first, i.e. what the production solvers do
seed_vdos_with_cdos = false

inp = arguments(
    a2f_file     = joinpath(pkg_dir, "test", "Nb", "Nb.a2F"),
    dos_file     = joinpath(pkg_dir, "test", "Nb", "Nb.dos"),
    Weep_file    = joinpath(pkg_dir, "test", "Nb", "Weep.dat"),
    outdir       = joinpath(@__DIR__, "output"),
    temps        = [6.0],
    cDOS_flag    = 0,
    include_Weep = 1,
    mu_flag      = 1,
    ind_smear    = 15,
    typEl        = 10000.0,
    # real-axis grid (only used for axis = :real)
    reOmega_c    = 2000.0,
    domega       = 0.4,
    mixing_beta  = 0.5,
    conv_thr     = 1e-3,
);

In [ ]:
# Create the output directory + log file, validate the inputs and read the files
# (same sequence as at the top of EliashbergSolver / RealAxisSolver).
# Note: createDirectory appends _1, _2, ... if outdir already exists.
strIsoME = IsoME.printIsoME()
inp, log_file, errorLogger = IsoME.createDirectory(inp, strIsoME)

realSolver = axis == :real
inp = IsoME.checkInput(inp, realSolver=realSolver)

inp, console, matval, ML_Tc = IsoME.InputParser(inp, log_file, mode=(realSolver ? 1 : 0))

IsoME.printFlagsAsText(inp, log_file, mode=(realSolver ? "realFreq" : "Matsubara"))

# unpack the material-specific values for inspection
(a2f_omega, a2f, dos_en, dos, Weep, dosef, idx_ef, ndos, BCS_gap, idxShiftcut) = matval

# what is available in this mode?
has_dos  = !isempty(dos)
has_Weep = !isnothing(Weep)

itemp = inp.temps[1]
β = 1 / (IsoME.kb * itemp)

println("mode           : ", inp.cDOS_flag == 1 ? "cDOS" : "vDOS", inp.include_Weep == 1 ? " + W" : " + μ*")
println("axis           : ", axis)
println("temperature    : ", itemp, " K")
println("ML-Tc estimate : ", round(ML_Tc, digits=2), " K")
println("BCS gap        : ", round(BCS_gap, digits=3), " meV")
println("μ*_AD          : ", inp.muc_AD)
println("μ*_ME          : ", inp.muc_ME)
println("μ              : ", inp.mu)
println("DOS read       : ", has_dos, has_dos ? "  (ε ∈ [$(round(dos_en[1], digits=1)), $(round(dos_en[end], digits=1))] meV, $(ndos) pts, N(εF) = $(round(dosef, digits=4)))" : "")
println("W read         : ", has_Weep, has_Weep ? "  ($(size(Weep, 1))×$(size(Weep, 2)))" : "")

## Step 2) The input data

Everything below is plotted *after* `InputParser`, i.e. after unit conversion, interpolation
onto the internal grids and the `encut` cut — this is what the solver actually sees, not the
raw file.

In [ ]:
# --- α²F(ω): always available ---
lambda_cum = [2 * trapz(a2f_omega[1:i], a2f[1:i] ./ a2f_omega[1:i]) for i in eachindex(a2f_omega)]
lambda_cum[1] = 0.0

pa = plot(a2f_omega, a2f, label="", xlabel="ω / meV", ylabel="α²F(ω)",
          title="Eliashberg spectral function")
pl = plot(a2f_omega, lambda_cum, label="", xlabel="ω / meV", ylabel="λ(ω)",
          title="cumulative λ = 2∫α²F/ω  (λ_tot = $(round(lambda_cum[end], digits=3)))")

# ω-cutoff used to convert μ*_AD -> μ*_ME, and the a2F support entering that formula
w_max = maximum(a2f_omega[a2f .> 0.01])
vline!(pa, [w_max], label="max ω (α²F > 0.01)", ls=:dash, c=:gray)

plot(pa, pl, layout=(1, 2), size=(950, 350))

In [ ]:
# --- N(ε): vDOS / μ-update / W modes only ---
if has_dos
    pd = plot(dos_en, dos, label="", xlabel="ε - ε_F / meV", ylabel="N(ε) / states/(meV·spin)",
              title="Electronic DOS")
    scatter!(pd, [dos_en[idx_ef]], [dosef], label="N(ε_F)", c=:red)

    # zoom on the window that actually matters for the Matsubara/ω' integrals
    zoom = 4 * inp.reOmega_c
    msk = abs.(dos_en) .<= zoom
    pz = plot(dos_en[msk], dos[msk], label="", xlabel="ε - ε_F / meV", ylabel="N(ε)",
              title="DOS near ε_F (±$(round(Int, zoom)) meV)")
    scatter!(pz, [dos_en[idx_ef]], [dosef], label="N(ε_F)", c=:red)

    # shiftcut window (χ / N_e integration range on the Matsubara axis)
    if idxShiftcut != -1
        vline!(pd, [dos_en[idxShiftcut[1]], dos_en[idxShiftcut[2]]], label="shiftcut", ls=:dash, c=:gray)
    end

    plot(pd, pz, layout=(1, 2), size=(950, 350))
else
    println("No DOS in this mode (cDOS + μ* without dos_file) - nothing to plot.")
end

In [ ]:
# --- W(ε,ε′): include_Weep = 1 only ---
if has_Weep
    pw = heatmap(dos_en, dos_en, Weep, xlabel="ε / meV", ylabel="ε′ / meV",
                 title="W(ε,ε′) / meV", c=:viridis)

    # the diagonal and the ε_F row: the entries that set the effective μ
    pcut = plot(dos_en, [Weep[i, i] for i in axes(Weep, 1)], label="W(ε,ε)",
                xlabel="ε / meV", ylabel="W / meV", title="W diagonal & ε_F row")
    plot!(pcut, dos_en, Weep[idx_ef, :], label="W(ε_F,ε′)")

    println("W(ε_F,ε_F)           = ", Weep[idx_ef, idx_ef], " meV")
    println("μ = N(ε_F)·W(ε_F,ε_F) = ", dosef * Weep[idx_ef, idx_ef])

    plot(pw, pcut, layout=(1, 2), size=(1000, 380))
else
    println("No W(ε,ε′) in this mode (include_Weep = 0) - nothing to plot.")
end

## Step 3) The Coulomb parameters

`InputParser` converts between μ, μ*_AD and μ*_ME with the Morel-Anderson-type formulas of
Pellegrini, *Ab initio methods for superconductivity* (`calcMucME` / `calcMucAD` / `calcMucs`
in `src/ReadIn.jl`):

$$\mu^*_{ME} = \frac{\mu^*_{AD}}{1 + \mu^*_{AD}\,\ln(\omega_{max}/\omega_c)}, \qquad
  \mu^*_{ME} = \frac{\mu}{1 + \mu\,\ln(\varepsilon_{typ}/\omega_c)}$$

with ω_c the frequency cutoff of the chosen axis (`imOmega_c` or `reOmega_c`) and ω_max the
largest ω where α²F > 0.01. Both curves below mark the value your `inp` actually ended up with.
The right panel is the one worth staring at: μ*_ME depends logarithmically on the cutoff, so it
must be re-derived whenever `imOmega_c` / `reOmega_c` changes — a cutoff far from the plateau
is a sign the pair is inconsistent.

`calcMucME` also clamps: if the formula returns μ*_ME < 0, > 0.8 or > 3·μ*_AD, it falls back to
`min(3·μ*_AD, 0.8)` and warns. That regime is shaded below.

In [ ]:
# ω-cutoff of the selected axis
omega_c = axis == :real ? inp.reOmega_c : inp.imOmega_c

muc_ME_from_AD(muAD, wc) = muAD ./ (1 .+ muAD .* log.(w_max ./ wc))
muc_ME_from_mu(mu, wc, etyp) = mu ./ (1 .+ mu .* log.(etyp ./ wc))

# --- left: μ*_ME vs its source parameter ---
if !isnothing(inp.mu) && !isnothing(inp.typEl)
    xs = range(0.01, max(0.6, 1.5 * inp.mu), length=200)
    p1 = plot(xs, muc_ME_from_mu(xs, omega_c, inp.typEl), label="μ*_ME(μ)",
              xlabel="μ", ylabel="μ*_ME", title="μ*_ME from μ  (ε_typ = $(inp.typEl) meV)")
    plot!(p1, xs, xs ./ (1 .+ xs .* log(inp.typEl / w_max)), label="μ*_AD(μ)", ls=:dash)
    vline!(p1, [inp.mu], label="μ = $(round(inp.mu, digits=3))", c=:gray, ls=:dot)
else
    xs = range(0.01, 0.3, length=200)
    p1 = plot(xs, muc_ME_from_AD(xs, omega_c), label="μ*_ME(μ*_AD)",
              xlabel="μ*_AD", ylabel="μ*_ME", title="μ*_ME from μ*_AD")
    plot!(p1, xs, xs, label="μ*_AD (identity)", ls=:dash, c=:gray)
    !isnothing(inp.muc_AD) && vline!(p1, [inp.muc_AD], label="μ*_AD = $(round(inp.muc_AD, digits=3))", c=:gray, ls=:dot)
end
!isnothing(inp.muc_ME) && hline!(p1, [inp.muc_ME], label="μ*_ME = $(round(inp.muc_ME, digits=3))", c=:red, ls=:dot)

# --- right: cutoff sensitivity of μ*_ME ---
wcs = range(max(w_max, 50.0), 5 * max(omega_c, w_max), length=300)
p2 = plot(xlabel="ω_c / meV", ylabel="μ*_ME", title="cutoff sensitivity  (axis = $(axis))")
if !isnothing(inp.mu) && !isnothing(inp.typEl)
    plot!(p2, wcs, muc_ME_from_mu(inp.mu, wcs, inp.typEl), label="μ*_ME(ω_c) from μ = $(round(inp.mu, digits=3))")
end
if !isnothing(inp.muc_AD)
    plot!(p2, wcs, muc_ME_from_AD(inp.muc_AD, wcs), label="μ*_ME(ω_c) from μ*_AD = $(round(inp.muc_AD, digits=3))")
    # clamp region of calcMucME
    hline!(p2, [min(3 * inp.muc_AD, 0.8)], label="clamp: min(3μ*_AD, 0.8)", c=:red, ls=:dash)
end
vline!(p2, [omega_c], label="ω_c = $(omega_c) meV", c=:gray, ls=:dot)
!isnothing(inp.muc_ME) && scatter!(p2, [omega_c], [inp.muc_ME], label="", c=:red)

if inp.include_Weep == 1
    println("include_Weep = 1: the Coulomb interaction comes from W(ε,ε′) directly;")
    println("μ*_ME is only used for the cDOS initial guess (derived from W(ε_F,ε_F)·N(ε_F) there).")
end

plot(p1, p2, layout=(1, 2), size=(1000, 380))

## Step 4) One self-consistency step

The cell below is the body of the solver's iteration loop, executed exactly once from the
initial guess (Z = 1, Δ = BCS gap, χ = 0), for the mode selected in `inp`:

* **`axis = :matsubara`** mirrors `solve_eliashberg` (`src/TcSearch.jl`): Matsubara grid,
  λ(iω_n−iω_m) via `calcLambda`, then one `eliashberg_eqn` call + linear mixing.
* **`axis = :real`** mirrors `solve_realAxis_cDOS` / `solve_realAxis_vDOS`
  (`src/realAxisSolver.jl`): `precompute` for the kernels, the pole-anchored ω′ head/tail
  workspace, then one `realEliashbergEq` call + mixing.

Note the μ-update is deliberately *not* applied here (on the Matsubara axis the solver itself
only starts it at `i_it > 1`), so `fermi_level = 0`. Step 5 inspects f(μ) separately, built
from the quantities this step produced.

In [ ]:
# mixing factor of the first iteration, and the Coulomb ramp weight (i_it = 1)
beta_mix  = IsoME.mixing_parameter(inp, 1)
wgCoulomb = min(1, 1 / inp.nItFullCoul)
mix(old, new) = (1.0 - abs(beta_mix)) .* old .+ abs(beta_mix) .* new

fermi_level = 0.0
chi_out  = nothing      # χ(ω_n) / χ(ω), vDOS only
w_chi    = nothing      # its grid (real axis: separate, longer grid)
fmu      = nothing      # f(μ) closure, built below when mu_flag = 1

if axis == :matsubara
    # ---------------- Matsubara axis: mirror of solve_eliashberg ---------------- #
    M    = ceil(Int, (inp.imOmega_c / (π * IsoME.kb * itemp) - 1) / 2)
    wsi  = collect((2 .* (0:M) .+ 1) .* π .* IsoME.kb .* itemp)
    nsiw = length(wsi)

    if itemp < inp.sparseSamplingTemp
        sparse_sampling_flag = 1
        ind_mat_freq = IsoME.initSparseSampling(β, inp.imOmega_c, M)
    else
        sparse_sampling_flag = 0
        ind_mat_freq = collect(1:M+1)
    end
    println("Matsubara frequencies: ", length(ind_mat_freq), " / ", nsiw,
            sparse_sampling_flag == 1 ? "  (sparse sampling)" : "")

    lambdai = IsoME.calcLambda(itemp, M, a2f_omega, a2f)

    # --- initial guess, exactly as in solve_eliashberg ---
    znormip = ones(nsiw)
    shiftip = zeros(nsiw)
    if inp.include_Weep == 1
        deltaip = ones(ndos, nsiw) .* BCS_gap
        phicip  = -ones(ndos) .* 0.1
        phiphip = ones(nsiw) .* maximum([BCS_gap, 2 * phicip[1]])
    else
        deltaip = ones(nsiw) .* BCS_gap
    end

    # --- one iteration ---
    if inp.include_Weep == 1 && inp.cDOS_flag == 0
        new_data = IsoME.eliashberg_eqn(itemp, nsiw, wsi, ind_mat_freq, sparse_sampling_flag, lambdai,
                                        dosef, ndos, dos_en, dos, Weep, znormip, phiphip, phicip, shiftip,
                                        wgCoulomb, fermi_level, idxShiftcut)
        znormi = mix(znormip, new_data[1]); phiphi = mix(phiphip, new_data[2])
        phici  = mix(phicip,  new_data[3]); shifti = mix(shiftip, new_data[4])
        deltai = (phiphi' .+ phici) ./ znormi'          # ndos × nsiw
        delta_out, chi_out = deltai[idx_ef, :], shifti

    elseif inp.include_Weep == 1 && inp.cDOS_flag == 1
        new_data = IsoME.eliashberg_eqn(itemp, nsiw, wsi, ind_mat_freq, sparse_sampling_flag, lambdai,
                                        ndos, dos_en, dos, Weep, znormip, phiphip, phicip, idx_ef, wgCoulomb)
        znormi = mix(znormip, new_data[1]); phiphi = mix(phiphip, new_data[2]); phici = mix(phicip, new_data[3])
        deltai = (phiphi' .+ phici) ./ znormi'
        delta_out = deltai[idx_ef, :]

    elseif inp.cDOS_flag == 0
        new_data = IsoME.eliashberg_eqn(itemp, nsiw, wsi, ind_mat_freq, sparse_sampling_flag, lambdai,
                                        dos_en, dos, dosef, znormip, deltaip, shiftip, Float64(inp.muc_ME),
                                        fermi_level, wgCoulomb, idxShiftcut)
        znormi = mix(znormip, new_data[1]); deltai = mix(deltaip, new_data[2]); shifti = mix(shiftip, new_data[3])
        delta_out, chi_out = deltai, shifti

    else
        new_data = IsoME.eliashberg_eqn(itemp, nsiw, wsi, ind_mat_freq, sparse_sampling_flag, lambdai,
                                        deltaip, Float64(inp.muc_ME), wgCoulomb)
        znormi = mix(znormip, new_data[1]); deltai = mix(deltaip, new_data[2])
        delta_out = deltai
    end

    w_out, Z_out = wsi, znormi
    w_chi = wsi

    # --- f(μ) closure, as assembled in update_mu_own (src/MuUpdate.jl) ---
    if inp.mu_flag == 1 && inp.cDOS_flag == 0
        a, b = idxShiftcut[1], idxShiftcut[2]
        en_c, dos_c = dos_en[a:b], dos[a:b]
        Ne_nsc = trapz(en_c, 2 .* IsoME.fermiFcn(en_c, 0.0, itemp) .* dos_c)
        # vDOS+W: Δ(ε,iω_n) is a matrix -> cut to the shiftcut window; vDOS+μ*: row vector
        deltaf = size(deltai, 2) == 1 ? permutedims(deltai) : deltai[a:b, :]
        fmu = x -> IsoME.diff_Ne(x, Ne_nsc, itemp, wsi, en_c, dos_c, znormi, deltaf, shifti)
    end

elseif axis == :real
    # ---------------- Real axis: mirror of solve_realAxis_cDOS / _vDOS ---------------- #
    realAxisParameter = IsoME.precompute(β, inp, matval, console, log_file)
    (w_static, w_static_chi, _) = realAxisParameter

    state = IsoME.initial_real_axis_state(inp, realAxisParameter, BCS_gap)
    if inp.cDOS_flag == 0 && seed_vdos_with_cdos
        state = IsoME.initialize_real_axis_vDOS(itemp, inp, console["cDOS"], matval, realAxisParameter, log_file)
    end

    if inp.cDOS_flag == 1
        # μ*_ME for the Weep case is derived from W(ε_F,ε_F)·N(ε_F), as in solve_realAxis_cDOS
        muc_ME = if isnothing(inp.muc_ME) && inp.include_Weep == 1
            typEl = !isnothing(inp.typEl) ? inp.typEl : inp.efW
            mu_W  = Weep[idx_ef, idx_ef] * dosef
            mu_W / (1 + mu_W * log(typEl / inp.reOmega_c))
        else
            Float64(inp.muc_ME)
        end

        gap0 = real(state.delta[1])
        ws_ht = IsoME.build_cDOS_wprime_workspace(inp, realAxisParameter, gap0)
        pole  = IsoME.find_cDOS_pole(w_static, state.delta, gap0)
        pole >= ws_ht.wp_max && (ws_ht = IsoME.build_cDOS_wprime_workspace(inp, realAxisParameter, pole))
        IsoME.maybe_refresh_head!(ws_ht, [pole], w_static, nothing, inp.n_cheb)

        Z_new, delta_new = IsoME.realEliashbergEq(muc_ME, β, state.delta, ws_ht, w_static)
        Z_out     = mix(state.Z, Z_new)
        delta_out = mix(state.delta, delta_new)

    else
        eps_1, eps_2 = transpose(dos_en[1:end-1]), transpose(dos_en[2:end])
        dos_1, dos_2 = transpose(dos[1:end-1]), transpose(dos[2:end])
        electronic_spec = (eps_1, eps_2, dos_1, dos_2, eps_2 .- eps_1, dos_2 .- dos_1)

        Z_prev, chi_prev = state.Z, state.chi
        # φ is the primary sc variable; vDOS+W carries φ(ω,ε), vDOS+μ* only φ(ω) = Δ·Z
        phi_prev = if inp.include_Weep == 1
            isnothing(state.phi) ? repeat(state.delta .* state.Z, 1, length(dos_en)) : (state.phi isa Matrix ? state.phi : repeat(state.phi, 1, length(dos_en)))
        else
            isnothing(state.phi) ? (state.delta .* state.Z) : state.phi
        end
        fermi_level = state.fermi_level
        gap0 = real(state.delta[1])

        gridws = IsoME.build_wprime_workspace(inp, realAxisParameter, gap0)
        poles = if inp.include_Weep == 1
            IsoME.find_integrand_poles_vDOSW(gridws.wp_full, w_static, w_static_chi, Z_prev, phi_prev,
                                             chi_prev, fermi_level, dos_en, idx_ef, gap0)
        else
            IsoME.find_integrand_poles(gridws.wp_full, w_static, w_static_chi, Z_prev, state.delta,
                                       chi_prev, fermi_level, gap0)
        end
        maximum(poles) >= gridws.wp_max && (gridws = IsoME.build_wprime_workspace(inp, realAxisParameter, maximum(poles)))
        IsoME.maybe_refresh_head!(gridws, poles, w_static, w_static_chi, inp.n_cheb)
        w_prime = gridws.wp_full

        if inp.include_Weep == 1
            Z_new, chi_new, phi_new = IsoME.realEliashbergEq(β, Z_prev, phi_prev, chi_prev, gridws, w_static,
                                                             w_static_chi, dosef, dos_en, dos, Weep, idx_ef,
                                                             fermi_level, wgCoulomb, electronic_spec)
        else
            Z_new, chi_new, phi_new = IsoME.realEliashbergEq(Float64(inp.muc_ME), β, Z_prev, phi_prev, chi_prev,
                                                             gridws, w_static, w_static_chi, dosef, dos_en, dos,
                                                             fermi_level, electronic_spec, poles)
        end

        Z_out   = mix(Z_prev, Z_new)
        chi_out = mix(chi_prev, chi_new)
        phi_out = mix(phi_prev, phi_new)
        delta_out = (inp.include_Weep == 1 ? phi_out[:, idx_ef] : phi_out) ./ Z_out

        # --- f(μ) closure, as assembled in mu_update_real_axis (src/MuUpdate.jl) ---
        if inp.mu_flag == 1
            Ne_nsc = 2 .* trapz(dos_en, IsoME.fermiFcn(dos_en, 0.0, itemp) .* dos)
            Z_ongrid     = linear_interpolation(w_static, Z_prev, extrapolation_bc=Flat()).(w_prime)
            phi_ongrid   = IsoME.interp_phi_ongrid(w_static, phi_prev, w_prime)
            shift_ongrid = linear_interpolation(w_static_chi, chi_prev, extrapolation_bc=Flat()).(w_prime)
            dos_int = trapz(dos_en, dos)
            tanhw   = tanh.(β .* w_prime ./ 2)
            fmu = x -> IsoME.diff_Ne_realAxis(x, Ne_nsc, tanhw, w_prime, electronic_spec, dos_int,
                                              Z_ongrid, phi_ongrid, shift_ongrid)
        end
    end

    w_out = collect(w_static)
    w_chi = collect(w_static_chi)

else
    error("Unknown axis = $(axis); use :matsubara or :real")
end

println("\nafter 1 iteration (mixing β = ", beta_mix, "):")
println("  Z(ω→0)  = ", Z_out[1])
println("  Δ(ω→0)  = ", delta_out[1], " meV   (BCS initial guess: ", BCS_gap, " meV)")
isnothing(chi_out) || println("  χ(ω→0)  = ", chi_out[1], " meV")

## Step 5) The self-energy components after one step

Δ and Z are one step from the initial guess, so do not read a gap value off these. What they
*do* show: whether the grids resolve the structure (α²F features must be visible in Z), whether
the ω-cutoff is reached before the curves flatten out, and whether anything has gone non-finite.

In [ ]:
xlab = axis == :real ? "ω / meV" : "iω_n / meV"

pz = plot(w_out, real.(Z_out), label="Re Z", xlabel=xlab, ylabel="Z", title="Z")
axis == :real && plot!(pz, w_out, imag.(Z_out), label="Im Z")

pd = plot(w_out, real.(delta_out), label="Re Δ", xlabel=xlab, ylabel="Δ / meV", title="Δ")
axis == :real && plot!(pd, w_out, imag.(delta_out), label="Im Δ")
hline!(pd, [BCS_gap], label="BCS guess", ls=:dash, c=:gray)

ps = [pz, pd]
if !isnothing(chi_out)
    pc = plot(w_chi, real.(chi_out), label="Re χ", xlabel=xlab, ylabel="χ / meV", title="χ")
    axis == :real && plot!(pc, w_chi, imag.(chi_out), label="Im χ")
    push!(ps, pc)
else
    println("cDOS mode: no χ.")
end

# sanity check
for (nm, v) in (("Z", Z_out), ("Δ", delta_out), ("χ", chi_out))
    isnothing(v) && continue
    all(isfinite, v) || @warn "$nm contains non-finite values after one iteration!"
end

plot(ps..., layout=(1, length(ps)), size=(400 * length(ps), 350))

## Step 6) f(μ) — the μ-update root

With `mu_flag = 1` and vDOS, each iteration solves

$$f(\mu) = N_e^{nsc} - N_e^{sc}(\mu) = 0$$

for the chemical potential (`root_finding` in `src/MuUpdate.jl`, a Regula-Falsi bracketed
search starting from ε_F ± 50 meV and widening in 50 meV steps, up to 100 times / ±5 eV).

f(μ) is built here from the quantities of the single iteration above. What to look for:

* a **single sign change**, well inside the plotted window — that is the root the solver finds;
* a **negative slope**: `root_finding` errors out with *"The number of electrons decreases with
  increasing mu!"* if f(μ₁) > f(μ₀) at the starting bracket;
* **no root within ±5 eV** of ε_F means the μ-update will fail — usually a DOS/`shiftcut` problem.

In [ ]:
if isnothing(fmu)
    println("No μ-update in this configuration (needs mu_flag = 1 and cDOS_flag = 0) - nothing to plot.")
else
    # the window root_finding starts from: ε_F ± 50 meV
    mu_wndw = 50.0
    mus = range(fermi_level - 4 * mu_wndw, fermi_level + 4 * mu_wndw, length=61)
    fs  = [fmu(m) for m in mus]

    pf = plot(mus, fs, label="f(μ) = N_e^nsc - N_e^sc", xlabel="μ / meV", ylabel="Δ N_e",
              title="μ-update root function (T = $(itemp) K)")
    hline!(pf, [0.0], label="", c=:black, ls=:dash)
    vline!(pf, [fermi_level - mu_wndw, fermi_level + mu_wndw], label="initial bracket", c=:gray, ls=:dot)

    # locate the sign change, if the window contains one
    idx = findfirst(i -> fs[i] * fs[i+1] < 0, 1:length(fs)-1)
    if isnothing(idx)
        @warn "No sign change of f(μ) in the plotted window - widen it or check the DOS / shiftcut."
    else
        mu_root = IsoME.RegulaFalsi(fmu, fs[idx], fs[idx+1], mus[idx], mus[idx+1], 1e-3, 1e-6)
        scatter!(pf, [mu_root], [0.0], label="root: μ = $(round(mu_root, digits=3)) meV", c=:red)
        println("f(μ) root : ", mu_root, " meV")
    end
    fs[end] > fs[1] && @warn "f(μ) increases with μ - root_finding will reject this slope."

    plot(pf, size=(600, 380))
end

In [ ]:
# ----- Finish: flush and close the log file -----
inp.testMode || close(log_file)